# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/capstone/starter_A_readmission.ipynb)

# BME i9400 — Capstone starter A
## Thirty-Day Readmission after a Diabetes Hospitalisation

This notebook loads the data, applies the fixed split, and fits the baseline. Run it from top to
bottom before you change anything; it takes about a minute.

---
## 1 — The data

About 100,000 inpatient encounters of patients with diabetes at 130 US hospitals, 1999–2008, from
Strack et al., "Impact of HbA1c measurement on hospital readmission rates", *BioMed Research
International*, 2014. Distributed by the UCI Machine Learning Repository under CC BY 4.0.

- **A row of $X$**: one hospital encounter. Demographics (race, sex, age band), admission type and
  source, discharge disposition, length of stay, counts of laboratory tests, procedures, medications
  and diagnoses, counts of the patient's outpatient, emergency and inpatient visits in the preceding
  year, three ICD-9 diagnosis codes, two laboratory results (HbA1c, glucose), and 23 columns on
  diabetes medications.
- **$y$**: 1 if the patient was readmitted within 30 days of discharge.

> **ICD-9** is the coding system for diagnoses used in US hospitals until 2015. `250.xx` codes are
> diabetes; `428` is heart failure; codes beginning with `V` or `E` are supplementary.

The column descriptions are in the paper's Table 1, and the meaning of the numeric admission and
discharge codes is in `IDS_mapping.csv`, which is in the same download.

In [ ]:
import io, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read())) as z:
    raw = pd.read_csv(z.open("diabetic_data.csv"), na_values="?", low_memory=False)
    ids = z.read("IDS_mapping.csv").decode()
print(f"{len(raw)} encounters, {raw['patient_nbr'].nunique()} patients, {raw.shape[1]} columns")
raw.head()

---
## 2 — Encounters that cannot be followed by a readmission

Encounters that ended in death or in discharge to hospice are removed, as in Strack et al.: for these
patients a readmission is not possible, or not the outcome of interest.

> `discharge_disposition_id` 11, 19, 20, 21 are deaths; 13 and 14 are hospice.

In [ ]:
DEATH_OR_HOSPICE = [11, 13, 14, 19, 20, 21]
data = raw[~raw["discharge_disposition_id"].isin(DEATH_OR_HOSPICE)].reset_index(drop=True)
y = (data["readmitted"] == "<30").astype(int).values
groups = data["patient_nbr"].values
print(f"{len(data)} encounters remain; readmitted within 30 days: {y.mean():.3f}")

---
## 3 — The fixed split

The split is **by patient**: all encounters of a patient are in the same set. 20% of the patients form
the test set, and 25% of the remaining patients the validation set. Do not change this cell.

> Splitting by encounter would place encounters of the same patient in the training and the test
> sets. The model could then recognise a patient rather than predict an outcome, and the test result
> would be optimistic (Meeting 1).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

dev, test = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=9400).split(data, y, groups))
tr, va    = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=9400).split(dev, y[dev], groups[dev]))
train, val = dev[tr], dev[va]

assert not (set(groups[train]) & set(groups[test])) and not (set(groups[val]) & set(groups[test]))
for name, ix in [("training", train), ("validation", val), ("test", test)]:
    print(f"{name:11s} {len(ix):6d} encounters  {len(set(groups[ix])):6d} patients  readmitted {y[ix].mean():.3f}")

---
## 4 — The baseline

L2-penalised logistic regression ($C = 1$) on a fixed set of 19 columns: the eight counts,
standardised, and eleven categorical columns, one-hot encoded with categories seen fewer than 50
times grouped together. The diagnosis codes and the individual medication columns are not used.

> The **primary metric** is the area under the ROC curve (AUROC, Meeting 9). Because only 11% of
> encounters are positive, also report the **average precision**, the area under the
> precision–recall curve, whose value for a classifier that ignores the features equals the
> prevalence.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

counts = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
          "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
categories = ["race", "gender", "age", "admission_type_id", "discharge_disposition_id", "admission_source_id",
              "A1Cresult", "max_glu_serum", "insulin", "change", "diabetesMed"]
X_base = data[counts + categories].copy()
X_base[categories] = X_base[categories].astype(str)

baseline = make_pipeline(
    ColumnTransformer([("counts", StandardScaler(), counts),
                       ("categories", OneHotEncoder(handle_unknown="ignore", min_frequency=50), categories)]),
    LogisticRegression(C=1.0, max_iter=2000))
baseline.fit(X_base.iloc[train], y[train])

p_val_base = baseline.predict_proba(X_base.iloc[val])[:, 1]
print(f"baseline, validation:  AUROC {roc_auc_score(y[val], p_val_base):.3f}   "
      f"average precision {average_precision_score(y[val], p_val_base):.3f}   (prevalence {y[val].mean():.3f})")

---
## 5 — The test set and the bootstrap

> **The test set is used once.** The cell below defines the test-set evaluation and the bootstrap
> confidence interval but does not run them. Run them at the end of the project, for your final model
> and for the baseline only, after every other decision has been made on validation data.

In [ ]:
def bootstrap_difference(metric, y, pred_new, pred_base, groups=None, n_boot=2000, seed=0):
    """95% bootstrap confidence interval for metric(y, pred_new) - metric(y, pred_base).

    Resamples whole groups (patients, records, abstracts) with replacement when `groups` is given,
    otherwise single examples. Both models are scored on the same resample, so the interval is for
    the paired difference (Meeting 20).
    """
    rng = np.random.default_rng(seed)
    y, pred_new, pred_base = np.asarray(y), np.asarray(pred_new), np.asarray(pred_base)
    if groups is None:
        groups = np.arange(len(y))
    uniq, inv = np.unique(np.asarray(groups), return_inverse=True)
    members = [np.flatnonzero(inv == k) for k in range(len(uniq))]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        diffs.append(metric(y[idx], pred_new[idx]) - metric(y[idx], pred_base[idx]))
    point = metric(y, pred_new) - metric(y, pred_base)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return point, lo, hi


def evaluate_on_test(p_test_new, p_test_base):
    """Primary metric of both models on the test set, and the 95% CI for the difference, resampling patients."""
    for name, p in [("final model", p_test_new), ("baseline", p_test_base)]:
        print(f"{name:12s} AUROC {roc_auc_score(y[test], p):.3f}   average precision {average_precision_score(y[test], p):.3f}")
    d, lo, hi = bootstrap_difference(roc_auc_score, y[test], p_test_new, p_test_base, groups=groups[test])
    print(f"difference in AUROC {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]")

# At the end of the project, and only then:
# evaluate_on_test(final_model.predict_proba(X_final.iloc[test])[:, 1], baseline.predict_proba(X_base.iloc[test])[:, 1])

---
## 6 — From here

The requirements, deliverables, and grading are in the capstone brief,
[`capstone/README.md`](https://github.com/dmochow/BME-i9400-2026/blob/main/capstone/README.md). In
short: reproduce the baseline above; search at least two further model families with every choice
made on validation data; evaluate the final model and the baseline once on the test set, with a 95%
bootstrap confidence interval for the difference; analyse the errors; write the model card.

Make a copy of this notebook (`File ▸ Save a copy in Drive`) and build your study in it.

Directions specific to this dataset: gradient boosting (`HistGradientBoostingClassifier`, Meeting 19)
with the categorical columns handled natively; the three diagnosis codes, grouped into clinical
categories; the medication columns; and the class imbalance. Expect modest gains: published models on
these data reach AUROCs around 0.65 to 0.70, so the confidence interval for the difference matters.